In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import copernicusmarine as cm

sys.path.insert(0, ".")
from src.barrier_layers import compute_segment_gradient, first_depth_below_threshold

In [33]:
date_string = '2023-12-30'

In [ ]:
ds = cm.open_dataset(
    dataset_id="cmems_mod_glo_phy-all_my_0.25deg_P1D-m",
    dataset_version="202311",
    variables=["mlotst_glor", "thetao_glor", "so_glor"],
    minimum_longitude=-180, maximum_longitude=179.75,
    minimum_latitude=-80,  maximum_latitude=90,
    start_datetime=date_string + "T00:00:00",
    end_datetime=date_string + "T23:59:59", 
    minimum_depth=0.5057600140571594, maximum_depth=508.639892578125,
    coordinates_selection_method="strict-inside",
    # username="dbrisaro@gmail.com",
    # password="Clarita1905.!!",
)

INFO - 2025-10-07T22:12:31Z - Selected dataset version: "202311"
INFO - 2025-10-07T22:12:31Z - Selected dataset part: "default"
INFO - 2025-10-07T22:12:31Z - Selected dataset part: "default"


In [ ]:
point_longitude = 55.0
point_latitude = -10.0
threshold = -0.1

ds_point = ds.isel(time=0).sel(latitude=point_latitude).sel(longitude=point_longitude)
lat = ds_point["latitude"].values
lon = ds_point["longitude"].values

mld = ds_point["mlotst_glor"].values
salinity_profile = ds_point["so_glor"].values
temperature_profile = ds_point["thetao_glor"].values
depth = ds_point["depth"].values

depth_mid, gradient = compute_segment_gradient(temperature_profile, depth)
ild, grad_at_ild = first_depth_below_threshold(gradient, depth_mid, threshold=threshold)

if ild is not None:
    bld = ild - mld

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = plt.axes([0.05, 0.05, 0.4, 0.9])
bx = ax.twiny()

ax.plot(temperature_profile, -depth, ".", color="salmon", markersize=3, linewidth=0.5, linestyle="-")
bx.plot(salinity_profile, -depth, ".", color="black", markersize=3, linewidth=0.5, linestyle="-")

if ild is not None:
    ax.axhline(y=-ild, color="lightblue", linewidth=1.5, linestyle="--")
    ax.text(temperature_profile.min(), -ild + 10, f"ILD: {ild:.1f}m", va="center", ha="left", color="lightblue", fontsize=9)
    ax.set_title(
        f"Vertical profile — Lat: {lat:.1f}°N, Lon: {lon:.1f}°E\n"
        f"Threshold: {threshold:.1f}°C/m  |  MLD: {mld:.1f}m  ILD: {ild:.1f}m  BLD: {bld:.1f}m",
        fontsize=11, pad=20,
    )
else:
    ax.set_title(
        f"Vertical profile — Lat: {lat:.1f}°N, Lon: {lon:.1f}°E\n"
        f"Threshold: {threshold:.1f}°C/m  |  MLD: {mld:.1f}m  ILD: undefined",
        fontsize=11, pad=20,
    )

ax.axhline(y=-mld, color="orange", linewidth=1.5, linestyle="--")
ax.text(temperature_profile.min(), -mld + 10, f"MLD: {mld:.1f}m", va="center", ha="left", color="orange", fontsize=9)

ax.set_xlabel("Temperature (°C)", color="salmon")
bx.set_xlabel("Salinity (psu)", color="black")
ax.set_ylabel("Depth (m)")
ax.set_ylim([-500, 0])
bx.set_xlim([34.65, 35.4])
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.spines["bottom"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", colors="salmon")
ax.spines["top"].set_color("salmon")
bx.xaxis.set_ticks_position("top")
bx.xaxis.set_label_position("top")
bx.spines["bottom"].set_visible(False)
bx.spines["right"].set_visible(False)
bx.spines["top"].set_position(("outward", 40))
bx.spines["top"].set_color("black")
bx.tick_params(axis="x", colors="black")
bx.xaxis.label.set_color("black")

cx = plt.axes([0.5, 0.05, 0.4, 0.9])
cx.plot(gradient, -depth_mid, ".", color="darkgrey", markersize=3, linewidth=0.5, linestyle="-")
cx.set_title("Temperature gradient", fontsize=11, pad=20)
cx.set_xlabel("dT/dz (°C/m)")
cx.set_ylim([-500, 0])
cx.xaxis.set_ticks_position("top")
cx.xaxis.set_label_position("top")
cx.spines["bottom"].set_visible(False)
cx.spines["right"].set_visible(False)
cx.axvline(x=threshold, color="black", linewidth=0.5, linestyle="-.")
cx.set_xlim([-0.3, 0])

date_str = ds.time.values[0].astype("datetime64[D]").astype(str)
fig.savefig(
    f"displays/profile_{date_str}_lat_{lat:.1f}_lon_{lon:.1f}_threshold_{threshold:.1f}.png",
    bbox_inches="tight", dpi=400,
)